In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
np.random.seed(42)

In [3]:
tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_set = datasets.FashionMNIST(root="./data", train=True,  download=True, transform=tf)
val_set   = datasets.FashionMNIST(root="./data", train=False, download=True, transform=tf)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False)

100%|██████████| 26.4M/26.4M [00:01<00:00, 16.2MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 433kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 8.41MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 43.4MB/s]


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

print("Using device:", device)

# Model
model = nn.Sequential(
    nn.Flatten(),

    nn.Linear(784, 256),
    nn.BatchNorm1d(256),
    nn.ReLU(),
    nn.Dropout(p=0.3),

    nn.Linear(256, 128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Dropout(p=0.3),

    nn.Linear(128, 64),
    nn.BatchNorm1d(64),
    nn.ReLU(),

    nn.Linear(64, 10)
).to(device)

# Print model architecture
print(model)

# --------------------------------------------------
# Loss, Optimizer, Scheduler
# --------------------------------------------------
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

epochs = 15

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs
)

# --------------------------------------------------
# Metric storage
# --------------------------------------------------
train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

# --------------------------------------------------
# Helper function
# --------------------------------------------------
def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = running_loss / total
    accuracy = 100 * correct / total

    return avg_loss, accuracy

# --------------------------------------------------
# Training loop
# --------------------------------------------------
for epoch in range(epochs):

    # ---------------- TRAIN ----------------
    model.train()

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # 1. Zero gradients
        optimizer.zero_grad()

        # 2. Forward pass
        outputs = model(images)

        # 3. Compute loss
        loss = criterion(outputs, labels)

        # 4. Backpropagation
        loss.backward()

        # 5. Update weights
        optimizer.step()

    # Step LR scheduler once per epoch
    scheduler.step()

    # ---------------- EVALUATION ----------------
    train_loss, train_acc = evaluate(
        model, train_loader, criterion, device
    )

    val_loss, val_acc = evaluate(
        model, val_loader, criterion, device
    )

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(
        f"Epoch [{epoch+1}/{epochs}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Acc: {val_acc:.2f}%"
    )

best_val_acc = max(val_accuracies)
best_epoch = val_accuracies.index(best_val_acc) + 1

print(f"\nBest Validation Accuracy: {best_val_acc:.2f}%")
print(f"Achieved at Epoch: {best_epoch}")

epochs_range = range(1, epochs + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, train_losses, label="Train Loss")
axes[0].plot(epochs_range, val_losses, label="Validation Loss")
axes[0].set_title("Loss vs Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(epochs_range, train_accuracies, label="Train Accuracy")
axes[1].plot(epochs_range, val_accuracies, label="Validation Accuracy")
axes[1].set_title("Accuracy vs Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].legend()

plt.tight_layout()
plt.show()

Using device: cpu
Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=256, bias=True)
  (2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (3): ReLU()
  (4): Dropout(p=0.3, inplace=False)
  (5): Linear(in_features=256, out_features=128, bias=True)
  (6): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (7): ReLU()
  (8): Dropout(p=0.3, inplace=False)
  (9): Linear(in_features=128, out_features=64, bias=True)
  (10): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (11): ReLU()
  (12): Linear(in_features=64, out_features=10, bias=True)
)
Epoch [1/15] | Train Loss: 0.3629 | Val Loss: 0.4071 | Train Acc: 86.87% | Val Acc: 85.23%
Epoch [2/15] | Train Loss: 0.3172 | Val Loss: 0.3674 | Train Acc: 88.68% | Val Acc: 86.64%
Epoch [3/15] | Train Loss: 0.3022 | Val Loss: 0.3614 | Train Acc: 88.80% | Val Acc: 86.87%
Epoch [4/15] | Train Loss: 0.2743 | 

The training and validation loss curves both decrease steadily across epochs, while the accuracy curves rise and begin to plateau toward the end of training. The validation metrics stay fairly close to the training metrics, suggesting that dropout and weight decay help limit overfitting. The best validation accuracy achieved was approximately 89–90%, which is consistent with expectations for this architecture.